In [2]:
from anthropic import Anthropic
from dotenv import load_dotenv
import json
from IPython.display import Markdown

load_dotenv()
client  = Anthropic()

model = "claude-sonnet-4-5"
temperature = 0.1
max_tokens = 1024
stop_sequences = []
system_prompt = ""
tools = []


In [3]:
def chat(message: str, temperature: float, system_prompt: str = "", stop_sequences: list = []):
    chat_message = [{"role": "user", "content": message}]
    response = client.messages.create(
        model=model, max_tokens=max_tokens, messages=chat_message,
        system=system_prompt, temperature=temperature, stop_sequences=stop_sequences
    )
    reply = response.content[0].text
    return reply

In [4]:
# Case 1 : Incomplete Prompt
def run_sentiment_analysis(test_case):
    prompt = f"""
        Determine the sentiment in the following tweet:
        {test_case["tweet_content"]}
        """
    output = chat(prompt, temperature=temperature, stop_sequences=stop_sequences)
    return output    

In [5]:
# Case 2 : Engineered Prompt

# Clear, Direct, Include Guidelines, Include common failure case examples, Output qualities
# Use XML tags to add structure

def run_sentiment_analysis_improved(test_case):
    prompt = f"""
        ## Identify the sentiment of the following tweet:
        {test_case["tweet_content"]}

        ## Generate output as a json with keys : 
        - sentiment (string value "postive", "negative", "neutral")
        - confidence_score (weigth of the sentiment in the text)
        - comments (cautionary note with maximum 10 words)

        ## Example of an input tweet with output:
        <input_tweet>
        I must say, the new 'smart' traffic signals installed at the junction have completely revolutionised my commute. Now, instead of waiting 2 minutes for a green light, I wait 5, but at least the blinking patterns are visually soothing. Truly a masterclass in urban planning.
        </input_tweet>
        <output>
        {{
            "sentiment": "negative", 
            "confidence_score": 0.95, 
            "comments": "Sarcasm detected; praise masks increased wait times."
        }}
        </output>

        """
    output = chat(prompt, temperature=temperature, stop_sequences=stop_sequences)
    return output    


In [6]:
# Read input dataset
tweets_list = []
with open("dataset2.json", "r") as f:
    tweets_json = json.load(f)
    for tweet in tweets_json:
        tweets_list.append(tweet)
# print(tweets_list)

model_outputs = []
for tweet in tweets_list:
    # print(tweet)
    current_tweet = {}
    current_tweet["input_tweet"] = tweet
    # current_tweet["model_response"] = run_sentiment_analysis(tweet)
    current_tweet["model_response"] = run_sentiment_analysis_improved(tweet)
    model_outputs.append(current_tweet)
    


In [9]:
from IPython.display import display, Markdown

for i, item in enumerate(model_outputs):
    display(Markdown(f"**Tweet {i+1}:** {item['input_tweet']['tweet_content']}"))
    display(Markdown(item['model_response']))

**Tweet 1:** What a stunning victory for Team India! Chasing down 350 against Australia in the 3rd ODI, with Kohli smashing a century. Absolutely electrifying atmosphere at the stadium!

```json
{
    "sentiment": "positive",
    "confidence_score": 0.98,
    "comments": "Genuine enthusiasm expressed; no sarcasm or negativity detected."
}
```

**Tweet 2:** This relentless heatwave in Delhi is unbearable. Power outages are making it worse, and there's no relief in sight for the next 48 hours. Staying indoors feels like a sauna. Please stay safe, everyone.

```json
{
    "sentiment": "negative",
    "confidence_score": 0.92,
    "comments": "Expresses distress about heatwave and power outages."
}
```

**Tweet 3:** The Ministry of Electronics and IT has officially released the draft framework for the National Data Governance Policy today. Public consultation is open until the end of next month for stakeholder feedback.

```json
{
    "sentiment": "neutral",
    "confidence_score": 0.92,
    "comments": "Factual announcement with no evaluative or emotional language."
}
```